# Automated Attendance Cron Job (Smart Schedule)
This notebook triggers immediately upon execution, and then automatically schedules itself for the exact times of your college periods (with a 1-minute delay).

**Schedule:**
- 1st Period: 9:01 AM
- 2nd Period: 9:51 AM
- 3rd Period: 10:56 AM
- 4th Period: 12:36 PM
- 5th Period: 1:26 PM
- 6th Period: 2:16 PM
- 7th Period: 3:56 PM

In [ ]:
!pip install pytz requests

In [ ]:
import time
import requests
import datetime
import pytz
import json

# ==========================================
# IMPORTANT: UPDATE THIS URL TO YOUR ACTUAL VERCEL URL
VERCEL_URL = "https://web-qr-blue.vercel.app/run-attendance"
# ==========================================

IST = pytz.timezone('Asia/Kolkata')

# Schedule with a 1-minute delay as requested
# Format: (hour, minute) in 24-hour time
SCHEDULE = [
    (9, 1),   # 1st Period: 9:00 AM + 1m
    (9, 51),  # 2nd Period: 9:50 AM + 1m
    (10, 56), # 3rd Period: 10:55 AM + 1m
    (12, 36), # 4th Period: 12:35 PM + 1m
    (13, 26), # 5th Period: 1:25 PM + 1m
    (14, 16), # 6th Period: 2:15 PM + 1m
    (15, 56)  # 7th Period: 3:55 PM + 1m
]

def trigger_attendance():
    print(f"\n[{datetime.datetime.now(IST).strftime('%Y-%m-%d %I:%M:%S %p')}] Triggering Attendance...")
    try:
        response = requests.get(VERCEL_URL, stream=True)
        
        if response.status_code != 200:
            print(f"❌ ERROR: Server returned {response.status_code}")
            print("Please make sure VERCEL_URL is correct!")
            return
            
        for line in response.iter_lines():
            if line:
                decoded_line = line.decode('utf-8')
                if decoded_line.startswith('data:'):
                    try:
                        log_data = json.loads(decoded_line[5:].strip())
                        msg = log_data.get('message', '')
                        if 'success' in msg.lower() or 'recorded' in msg.lower() or 'already' in msg.lower():
                            print(f"✅ {msg}")
                        elif 'error' in msg.lower() or 'failed' in msg.lower() or 'expired' in msg.lower():
                            print(f"❌ {msg}")
                        else:
                            print(f"-> {msg}")
                    except Exception:
                        pass
    except Exception as e:
        print(f"Error triggering attendance: {e}")

def get_next_period_time(now):
    """Finds the very next scheduled period from the current time."""
    for (hour, minute) in SCHEDULE:
        target = now.replace(hour=hour, minute=minute, second=0, microsecond=0)
        if target > now:
            return target
            
    # If no periods left today, return the first period of tomorrow
    first_hour, first_minute = SCHEDULE[0]
    tomorrow = now + datetime.timedelta(days=1)
    return tomorrow.replace(hour=first_hour, minute=first_minute, second=0, microsecond=0)

print("🎓 Smart Attendance Automator Started.")
print(f"Target URL: {VERCEL_URL}")

# 1. Trigger immediately upon starting the notebook! (Handles late starts like 10 AM)
print("\n[INIT] Triggering initial attendance immediately as requested...")
trigger_attendance()

# 2. Enter the infinite loop for all upcoming scheduled periods
while True:
    now = datetime.datetime.now(IST)
    next_time = get_next_period_time(now)
    
    sleep_seconds = (next_time - now).total_seconds()
    
    if sleep_seconds > 3600:
        print(f"\nWaiting {sleep_seconds/3600:.1f} hours for the next scheduled period at {next_time.strftime('%I:%M %p')}...")
    else:
        print(f"\nWaiting {sleep_seconds/60:.0f} minutes for the next scheduled period at {next_time.strftime('%I:%M %p')}...")
    
    # Sleep until exactly the next period time
    while True:
        current_now = datetime.datetime.now(IST)
        if current_now >= next_time:
            break
        time.sleep(min((next_time - current_now).total_seconds(), 60))
        
    # Trigger!
    trigger_attendance()
